# GoldAgent - 黄金市场分析 Notebook

本 Notebook 用于探索性分析和模型开发

## 目录
1. 环境配置
2. 数据获取
3. 数据处理
4. 技术指标计算
5. 市场分析
6. 交易信号生成
7. 可视化展示

## 1. 环境配置和导入

In [ ]:
import sys
from pathlib import Path

# 添加项目路径
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import logging
from datetime import datetime, timedelta

# 配置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ 环境配置完毕")

In [ ]:
# 导入自定义模块
from src.config_loader import get_config
from src.data_fetcher import DataFetcher
from src.data_processor import DataProcessor
from src.indicator_engine import IndicatorEngine
from src.agent_brain import AgentBrain
from src.visualizer import Visualizer

# 加载配置
config = get_config()
print(f"✓ 配置已加载")
print(f"  - 模型: {config.get('agent.model')}")
print(f"  - 时区: {config.get('data.processing.timezone')}")

## 2. 数据获取

In [ ]:
# 初始化数据获取器
fetcher = DataFetcher()

print("获取国际黄金数据...")
intl_data = fetcher.fetch_international_gold(
    symbol=config.get('data.international.symbol', 'GC=F'),
    period=config.get('data.international.period', '1y')
)
print(f"✓ 国际黄金数据: {len(intl_data)} 行")
print(intl_data.head())

In [ ]:
print("获取国内黄金数据...")
dom_data = fetcher.fetch_domestic_gold(
    symbol=config.get('data.domestic.symbol', '沪金连续')
)
print(f"✓ 国内黄金数据: {len(dom_data)} 行")
print(dom_data.head())

## 3. 数据处理

In [ ]:
# 初始化处理器
processor = DataProcessor(
    timezone=config.get('data.processing.timezone', 'Asia/Shanghai')
)

# 执行处理流程
intl_processed, dom_processed = processor.process_pipeline(
    intl_data.copy(),
    dom_data.copy(),
    config=config.get('data.processing', {})
)

print("✓ 数据处理完成")
print(f"  - 国际数据: {len(intl_processed)} 行")
print(f"  - 国内数据: {len(dom_processed)} 行")

In [ ]:
# 计算溢价
merged_data = processor.calculate_premium(intl_processed, dom_processed)
print("✓ 溢价计算完成")
print(merged_data[['close', 'close_2', 'premium_pct']].tail(10))

## 4. 技术指标计算

In [ ]:
# 初始化指标引擎
engine = IndicatorEngine()

# 计算所有指标
data_with_indicators = engine.calculate_all_indicators(
    merged_data.copy(),
    config=config.get('indicator', {})
)

print("✓ 技术指标计算完成")
print(f"  列数: {data_with_indicators.shape[1]}")
print(f"  行数: {data_with_indicators.shape[0]}")

In [ ]:
# 查看计算结果
indicator_cols = ['ma5', 'ma20', 'ma60', 'rsi', 'obv']
print(data_with_indicators[indicator_cols].tail(10))

## 5. 交易信号生成

In [ ]:
# 生成交易信号
data_with_signals = engine.generate_signals(data_with_indicators.copy(), strategy='ma_cross')

print("✓ 交易信号生成完成")

# 统计信号
buy_signals = (data_with_signals['signal'] == 1).sum()
sell_signals = (data_with_signals['signal'] == -1).sum()

print(f"  - 买入信号: {buy_signals}")
print(f"  - 卖出信号: {sell_signals}")

In [ ]:
# 显示最近的交易信号
signals_data = data_with_signals[['close', 'ma5', 'ma20', 'signal']].tail(20)
print(signals_data)

## 6. 市场分析

In [ ]:
# 初始化 Agent Brain
brain = AgentBrain()

# 获取最新数据
latest = data_with_indicators.iloc[-1]

# 构建市场上下文
market_context = brain.format_market_context(
    current_price=latest.get('close', 0),
    ma_5=latest.get('ma5', 0),
    ma_20=latest.get('ma20', 0),
    ma_60=latest.get('ma60', 0),
    rsi=latest.get('rsi', 50),
    premium=latest.get('premium_pct', 0),
    volume=latest.get('volume', 0)
)

print("✓ 市场上下文构建完成")
print(f"  - 当前价格: ${latest.get('close', 0):.2f}")
print(f"  - MA5: ${latest.get('ma5', 0):.2f}")
print(f"  - MA20: ${latest.get('ma20', 0):.2f}")
print(f"  - RSI: {latest.get('rsi', 0):.2f}")
print(f"  - 溢价率: {latest.get('premium_pct', 0):.2f}%")

In [ ]:
# 执行综合分析
analysis = brain.analyze(
    market_data=market_context,
    indicators=market_context['momentum'],
    task='analyze'
)

print("✓ 市场分析完成")
print(f"\n关键发现:")
for finding in analysis.get('key_findings', []):
    print(f"  - {finding}")

## 7. 可视化展示

In [ ]:
# 初始化可视化器
visualizer = Visualizer(output_dir="reports")

# 绘制 K 线 + MA 图
recent_data = data_with_indicators.iloc[-100:]  # 最近100条
fig = visualizer.plot_with_ma(
    recent_data,
    ma_columns=['ma5', 'ma20', 'ma60'],
    title="Gold Price with Moving Averages (Recent 100 Days)"
)
fig.show()

In [ ]:
# 绘制综合图表（包含买卖点）
signals_recent = data_with_signals.iloc[-100:]
fig = visualizer.plot_combined(
    recent_data,
    ma_columns=['ma5', 'ma20'],
    indicators_to_plot={'rsi': 'RSI'},
    buy_sell_signals=signals_recent,
    title="Gold Market Analysis with Trading Signals"
)
fig.show()

In [ ]:
# 绘制溢价分析
fig = visualizer.plot_premium_comparison(
    recent_data,
    intl_col='close',
    dom_col='close_2',
    premium_col='premium_pct',
    title="Gold Premium Analysis (Domestic vs International)"
)
fig.show()

## 8. 报告生成

In [ ]:
# 生成分析报告
report = brain.generate_report(analysis)
print(report)

In [ ]:
# 保存报告
from pathlib import Path
from datetime import datetime

Path('reports').mkdir(exist_ok=True)

report_path = f"reports/analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"✓ 报告已保存: {report_path}")

## 数据统计总结

In [ ]:
# 最终统计
print("=" * 50)
print("分析统计总结")
print("=" * 50)
print(f"\n时间范围: {data_with_indicators.index[0].strftime('%Y-%m-%d')} ~ {data_with_indicators.index[-1].strftime('%Y-%m-%d')}")
print(f"\n价格统计:")
print(f"  - 最高: ${data_with_indicators['close'].max():.2f}")
print(f"  - 最低: ${data_with_indicators['close'].min():.2f}")
print(f"  - 平均: ${data_with_indicators['close'].mean():.2f}")
print(f"  - 当前: ${latest['close']:.2f}")
print(f"\n技术指标:")
print(f"  - RSI: {latest['rsi']:.2f}")
print(f"  - 成交量: {latest['volume']:.0f}")
print(f"  - 溢价率: {latest['premium_pct']:.2f}%")
print(f"\n交易信号:")
print(f"  - 买入信号: {buy_signals}")
print(f"  - 卖出信号: {sell_signals}")
print("\n" + "=" * 50)